# Symbolic network on MNIST 0 vs 1 — Milestone A

Implements the Milestone A target from `docs/research/hybrid_symbolic_networks.md`:

- `SymbolicNetwork` dataclass: **K=4 layer-1 feature trees + 1 layer-2 classifier tree**
- Network-aware GP: mutation operates on one tree slot within the network
- 2-class problem: **MNIST digit 0 vs digit 1** (binary, easiest contrast)
- **Success criterion**: TEST accuracy > 0.95

Per `tessera/experimental/symbolic_network.py`. This is Colab-runnable — installs tessera from GitHub if needed.

```
image (28x28)
  │
  ├─→ T_1(image) ─→ mean-pool ─→ f_0 ─┐
  ├─→ T_2(image) ─→ mean-pool ─→ f_1 ─┤
  ├─→ T_3(image) ─→ mean-pool ─→ f_2 ─├─→ T_clf(f_0,f_1,f_2,f_3) ─→ score ─→ sigmoid → class
  └─→ T_4(image) ─→ mean-pool ─→ f_3 ─┘
```

GP unit = the whole (4+1)-tree network. Mutation picks one slot per step.

## 1. Setup

If running locally with the repo already cloned, just skip the git clone cell.

In [ ]:
# Cell 1.1 — Clone repo if not already on path (Colab use).
import os, sys
REPO = 'tessera'
if not os.path.exists(REPO):
    !git clone --depth 1 https://github.com/davechendatascience/tessera.git
src_path = os.path.abspath(os.path.join(REPO, 'src'))
if src_path not in sys.path:
    sys.path.insert(0, src_path)
print('tessera src on path:', src_path)

In [ ]:
# Cell 1.2 — Imports.
import random
import time
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml

from tessera.experimental.symbolic_network import (
    SymbolicNetwork, NetworkGPConfig, run_network_gp,
    network_accuracy, evaluate_network,
)
from tessera.expression.tree import evaluate as eval_tree

print('tessera.experimental.symbolic_network loaded')

## 2. Load MNIST digits 0 vs 1

- Subset to digits 0 and 1 only
- Downsample 28×28 → 14×14 (2× block-mean) for GP speed
- 400 TRAIN / 200 TEST per class, stratified

In [ ]:
# Cell 2 — Load + preprocess MNIST 0 vs 1.
SEED = 2026
TARGET_A, TARGET_B = 0, 1   # the two classes
N_PER_TRAIN = 400
N_PER_TEST  = 200
IMG_SIZE = 14               # downsampled from 28

def downsample_2x(img):
    """28×28 → 14×14 via 2×2 block-mean."""
    return img.reshape(14, 2, 14, 2).mean(axis=(1, 3))

print('Fetching MNIST via sklearn (cached after first call)...')
t0 = time.time()
mnist = fetch_openml('mnist_784', version=1, as_frame=False, cache=True)
print(f'  done in {time.time()-t0:.1f}s')
X = mnist.data.reshape(-1, 28, 28).astype(np.float32) / 255.0
y = mnist.target.astype(int)

rng_data = np.random.default_rng(SEED)
idx_a = rng_data.permutation(np.where(y == TARGET_A)[0])
idx_b = rng_data.permutation(np.where(y == TARGET_B)[0])

tr_a, tr_b = idx_a[:N_PER_TRAIN], idx_b[:N_PER_TRAIN]
te_a, te_b = idx_a[N_PER_TRAIN:N_PER_TRAIN+N_PER_TEST], idx_b[N_PER_TRAIN:N_PER_TRAIN+N_PER_TEST]

tr_idx = np.concatenate([tr_a, tr_b])
te_idx = np.concatenate([te_a, te_b])
rng_data.shuffle(tr_idx); rng_data.shuffle(te_idx)

imgs_train_full = X[tr_idx]
labels_train = (y[tr_idx] == TARGET_B).astype(int)  # 0 if digit=0, 1 if digit=1
imgs_test_full  = X[te_idx]
labels_test  = (y[te_idx] == TARGET_B).astype(int)

imgs_train = np.stack([downsample_2x(im) for im in imgs_train_full], axis=0)
imgs_test  = np.stack([downsample_2x(im) for im in imgs_test_full], axis=0)

print(f'TRAIN: {imgs_train.shape}, labels = {np.bincount(labels_train)}')
print(f'TEST:  {imgs_test.shape}, labels = {np.bincount(labels_test)}')

In [ ]:
# Cell 2.1 — Visualize sample digits.
fig, axes = plt.subplots(2, 6, figsize=(8, 3))
for i in range(6):
    j0 = np.where(labels_train == 0)[0][i]
    j1 = np.where(labels_train == 1)[0][i]
    axes[0, i].imshow(imgs_train[j0], cmap='gray')
    axes[0, i].set_title(f'digit {TARGET_A}', fontsize=8)
    axes[0, i].axis('off')
    axes[1, i].imshow(imgs_train[j1], cmap='gray')
    axes[1, i].set_title(f'digit {TARGET_B}', fontsize=8)
    axes[1, i].axis('off')
plt.suptitle(f'Sample TRAIN digits ({IMG_SIZE}×{IMG_SIZE} downsampled)', fontsize=10)
plt.tight_layout()
plt.show()

## 3. Run the network-aware GP

Configuration tuned for Colab (small enough to finish in ~5 min):
- pop_size = 30 networks
- n_gens = 30
- K = 4 layer-1 features
- enable_2d = True (Measure2D ops available for layer-1 convolutional structure)

Each generation evaluates `pop_size` offspring + scores against TRAIN images. Best network is preserved via μ+λ survival.

In [ ]:
# Cell 3 — Run GP.
cfg = NetworkGPConfig(
    pop_size=30,
    n_gens=30,
    K=4,
    layer_1_max_depth=3,
    layer_2_max_depth=3,
    enable_2d=True,
    parsimony=0.002,
    tournament_size=3,
    crossover_rate=0.3,
    seed=SEED,
    early_stop_patience=12,
    verbose=True,
)

t0 = time.time()
best, history = run_network_gp(
    imgs_train, labels_train, cfg,
    imgs_test, labels_test,
)
elapsed = time.time() - t0

print()
print(f'Runtime: {elapsed:.1f}s')
print(f'Best TRAIN accuracy: {best.accuracy:.3f}')
print(f'Best TEST  accuracy: {network_accuracy(best.network, imgs_test, labels_test):.3f}')
print(f'Best complexity: {best.network.complexity}')

## 4. Training curves

Loss + accuracy per generation. TEST is evaluated on the best-of-generation network so its curve is noisier than TRAIN.

In [ ]:
# Cell 4 — Training curves.
gens = [h['gen'] for h in history]
losses = [h['best_loss'] for h in history]
tr_accs = [h['best_train_acc'] for h in history]
te_accs = [h['best_test_acc'] for h in history]
cxs    = [h['best_cx'] for h in history]

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
axes[0].plot(gens, losses, marker='o', lw=1)
axes[0].set_xlabel('generation'); axes[0].set_ylabel('best loss')
axes[0].set_title('Network loss per generation')
axes[0].grid(alpha=0.3)

axes[1].plot(gens, tr_accs, marker='o', lw=1, label='TRAIN')
axes[1].plot(gens, te_accs, marker='s', lw=1, label='TEST')
axes[1].axhline(0.95, color='red', ls='--', alpha=0.5, label='success bar (0.95)')
axes[1].axhline(0.80, color='gray', ls='--', alpha=0.5, label='single-tree baseline (0.80)')
axes[1].set_xlabel('generation'); axes[1].set_ylabel('accuracy')
axes[1].set_title('Accuracy per generation')
axes[1].set_ylim(0.4, 1.05)
axes[1].legend(loc='lower right', fontsize=8)
axes[1].grid(alpha=0.3)

axes[2].plot(gens, cxs, marker='o', lw=1, color='green')
axes[2].set_xlabel('generation'); axes[2].set_ylabel('total cx')
axes[2].set_title('Network complexity (sum of trees)')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 5. What did the network discover?

Print each tree slot's expression + visualize the layer-1 features applied to sample images. Interpretation question: do the discovered features map to human concepts (closed-loop vs vertical-stroke), or are they opaque pixel correlations?

In [ ]:
# Cell 5.1 — Print the network.
print(best.network)

In [ ]:
# Cell 5.2 — Visualize each layer-1 tree applied to a sample of each class.
def safe_eval_to_field(tree, image):
    """Evaluate tree on image; return 2D array (broadcast scalar if needed)."""
    try:
        out = eval_tree(tree, {'image': image}, fill_warmup=0.0)
        out = np.asarray(out, dtype=np.float64)
        if out.ndim == 0:
            out = np.full_like(image, float(out), dtype=np.float64)
        return out
    except Exception as e:
        return np.zeros_like(image)

samples_per_class = 2
K = best.network.K

j0_list = np.where(labels_test == 0)[0][:samples_per_class]
j1_list = np.where(labels_test == 1)[0][:samples_per_class]
sample_indices = list(j0_list) + list(j1_list)
sample_classes = [TARGET_A]*samples_per_class + [TARGET_B]*samples_per_class

fig, axes = plt.subplots(len(sample_indices), K+1, figsize=(2.0*(K+1), 2.0*len(sample_indices)))
if len(sample_indices) == 1:
    axes = axes[None, :]
for r, (idx, cls) in enumerate(zip(sample_indices, sample_classes)):
    img = imgs_test[idx]
    axes[r, 0].imshow(img, cmap='gray')
    axes[r, 0].set_title(f'digit {cls}', fontsize=9)
    axes[r, 0].axis('off')
    for k in range(K):
        field = safe_eval_to_field(best.network.layer_1_trees[k], img)
        axes[r, k+1].imshow(field, cmap='RdBu_r')
        axes[r, k+1].set_title(f'f_{k}', fontsize=9)
        axes[r, k+1].axis('off')
plt.suptitle('Layer-1 feature responses (before mean-pool)', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 5.3 — Feature distributions across TEST classes.
# For each f_k, plot the distribution of mean-pooled values across digit 0 vs digit 1.
# If a feature is informative, the two distributions should be visibly separated.

def feature_values(tree, images):
    vals = np.zeros(len(images), dtype=np.float64)
    for i, img in enumerate(images):
        out = safe_eval_to_field(tree, img)
        vals[i] = np.nanmean(out)
    return vals

fig, axes = plt.subplots(1, K, figsize=(3*K, 3))
if K == 1:
    axes = [axes]
for k in range(K):
    vals = feature_values(best.network.layer_1_trees[k], imgs_test)
    axes[k].hist(vals[labels_test==0], bins=20, alpha=0.6, label=f'digit {TARGET_A}', color='C0')
    axes[k].hist(vals[labels_test==1], bins=20, alpha=0.6, label=f'digit {TARGET_B}', color='C1')
    axes[k].set_title(f'f_{k}')
    axes[k].set_xlabel('mean-pooled value')
    axes[k].legend(fontsize=7)
plt.tight_layout()
plt.show()

## 6. Final scoreboard

Compare to the Milestone A success criterion and the existing single-tree baseline.

In [ ]:
# Cell 6 — Scoreboard.
tr_acc = best.accuracy
te_acc = network_accuracy(best.network, imgs_test, labels_test)

print('=' * 55)
print(f'Milestone A — MNIST {TARGET_A} vs {TARGET_B}, K={best.network.K}, pop={cfg.pop_size}, gens={cfg.n_gens}')
print('=' * 55)
print(f'  TRAIN accuracy:   {tr_acc:.3f}  ({int(tr_acc * len(labels_train))} / {len(labels_train)})')
print(f'  TEST accuracy:    {te_acc:.3f}  ({int(te_acc * len(labels_test))} / {len(labels_test)})')
print(f'  Network cx:       {best.network.complexity}')
print()
print(f'  Single-tree baseline (existing benchmark, 0-vs-rest): TEST 0.80')
print(f'  Milestone A success bar:                              TEST > 0.95')
print()
if te_acc > 0.95:
    print('  >>> SUCCESS: TEST accuracy exceeds Milestone A target.')
elif te_acc > 0.80:
    print('  >>> PROGRESS: TEST accuracy beats single-tree baseline but below 0.95 target.')
    print('      Try larger pop_size / n_gens, or K, or fewer training samples.')
else:
    print('  >>> NOT YET: TEST accuracy ≤ single-tree baseline.')
    print('      Architecture may need infrastructure refinement (see hybrid_symbolic_networks.md §5).')

## What's next

Per `docs/research/hybrid_symbolic_networks.md`:

- **Milestone B**: Add neural prepass (small CNN teacher → extract filters → convert to symbolic seed trees → inject into network initial population). This is Hybrid #2 — detect-then-seed at the network composition layer.
- **Milestone C**: Scale up to full 10-class MNIST with K=16, cross-entropy loss.
- **Milestone D**: Cross-domain validation (Fashion-MNIST, CIFAR-10 if scope allows).

If Milestone A's TEST accuracy is below 0.95 but above 0.80, the architecture is working but the search budget or K is too small. If it's at or below 0.80, refer to §5 of the research note for which infrastructure piece likely needs work.